In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q onnx onnxruntime ultralytics 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 53.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 7.3 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from time import perf_counter
import gc
import hashlib
import platform

import cv2
import numpy as np
import pandas as pd
import torch
import onnxruntime as ort
import ultralytics
import yaml

from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
DRIVE = Path("/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike")

MODEL_PT = DRIVE / "runs" / "yolo26n_rdd2022_baseline" / "weights" / "best.pt"
DATA_YAML = DRIVE / "configs" / "rdd2022_china_motorbike.yaml"
VAL_IMAGES = DRIVE / "val" / "images"

OUT = DRIVE / "out"
OUT.mkdir(parents=True, exist_ok=True)

if MODEL_PT.exists():
    weight_hash = hashlib.sha256(MODEL_PT.read_bytes()).hexdigest()
    print("Weights SHA256:", weight_hash)
else:
    print(f"Error: Weights file not found at {MODEL_PT}")

Weights SHA256: f2bfe936f87b9b7362baa7e5318ed55b91271f42ed5af79e4262736daa5f9d99


In [5]:
if DATA_YAML.exists():
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f) or {}
    
    current_yaml_path = data_config.get("path", "")
    if current_yaml_path != str(DRIVE):
        data_config["path"] = str(DRIVE)
        
        with open(DATA_YAML, 'w') as f:
            yaml.dump(data_config, f, sort_keys=False)
        print(f"Updated YAML path to: {DRIVE}")
        
        for split in ["train", "val"]:
            cache_file = DRIVE / split / "labels.cache"
            if cache_file.exists():
                cache_file.unlink()
                print(f"Deleted outdated cache: {cache_file.name}")
else:
    print(f"Warning: DATA_YAML not found at {DATA_YAML}")

In [6]:
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

all_paths = sorted(
    [p for p in VAL_IMAGES.rglob("*") if p.suffix.lower() in extensions],
    key=lambda p: p.relative_to(VAL_IMAGES).as_posix(),
)

rng = np.random.default_rng(42)
indices = rng.choice(
    len(all_paths),
    size=min(50, len(all_paths)),
    replace=False,
)

sample_paths = [all_paths[int(i)] for i in indices]
images = [cv2.imread(str(p)) for p in sample_paths]

print("Latency images:", len(images))
print("First three:", [p.name for p in sample_paths[:3]])

Latency images: 50
First three: ['China_MotorBike_000787.jpg', 'China_MotorBike_000947.jpg', 'China_MotorBike_000856.jpg']


In [7]:
IMGSZ = 640
WARMUP = 20
REPEATS = 3

COMMON = {
    "imgsz": IMGSZ,
    "rect": False,
    "iou": 0.70,
    "max_det": 300,
    "augment": False,
    "verbose": False,
}

def precision_args(fp16=False):
    if "quantize" in DEFAULT_CFG_DICT:
        return {"quantize": 16 if fp16 else 32}
    return {"half": fp16}


if DEFAULT_CFG_DICT.get("nms", False) is None:
    HEAD_ARGS = {"nms": False}
else:
    HEAD_ARGS = {"end2end": True}

print("FP32 arguments:", precision_args(False))
print("FP16 arguments:", precision_args(True))
print("Head arguments:", HEAD_ARGS)

FP32 arguments: {'quantize': 32}
FP16 arguments: {'quantize': 16}
Head arguments: {'nms': False}


**FP32 ONNX export**

In [8]:
export_args = {
    **HEAD_ARGS,
    **precision_args(False),
    "format": "onnx",
    "imgsz": IMGSZ,
    "batch": 1,
    "dynamic": False,
    "simplify": False,
    "opset": 17,
    "device": "cpu",
    "nms": False,
}

MODEL_ONNX = Path(
    YOLO(str(MODEL_PT)).export(**export_args)
).resolve()

assert MODEL_ONNX.is_file()
print("Exported:", MODEL_ONNX)

Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.23.2 opset 17...
ONNX: export success ✅ 2.3s, saved as '/content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.onnx' (9.3 MB)

Export complete (3.0s)
Results saved to /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/Vi

In [9]:
def synchronize(device):
    if device != "cpu":
        torch.cuda.synchronize()
        

def prepare_model(path, device="cpu", fp16=False):
    model = YOLO(str(path), task="detect")
    
    kwargs = {
        **COMMON,
        **HEAD_ARGS,
        **precision_args(fp16),
        "device": device,
        "conf": 0.25,
        "dnn": False,
        "save": False
    }
    
    for _ in range(WARMUP):
        model.predict(images[0], **kwargs)
    
    synchronize(device)
    backend = model.predictor.model
    print("Device:", backend.device, "| FP16:", backend.fp16)
    
    if Path(path).suffix == ".onnx":
        providers = backend.session.get_providers()
        print("Active providers:", providers)
        assert providers == ["CPUExecutionProvider"]

    return model, kwargs

**Measurement function**

In [10]:
def measure_latency(model, kwargs, name):
    predict_times = []
    inference_times = []
    
    for _ in range(REPEATS):
        for image in images:
            synchronize(kwargs["device"])
            started = perf_counter()
            
            result = model.predict(image, **kwargs)[0]
            synchronize(kwargs["device"])
            
            elapsed_ms = (perf_counter() - started) * 1000
            predict_times.append(elapsed_ms)
            inference_times.append(result.speed["inference"])
        
    
    raw = pd.DataFrame({
        "predict_ms": predict_times,
        "inference_ms": inference_times
    })
    raw.to_csv(OUT / f"{name}_latency.csv", index=False)
    return {
        "timed_calls": len(raw),
        "predict_mean_ms": float(np.mean(predict_times)),
        "predict_p95_ms": float(np.percentile(predict_times, 95)),
        "inference_mean_ms": float(np.mean(inference_times))
    }

In [11]:
def measure_accuracy(path, name, device="cpu", fp16=False):
    model = YOLO(str(path), task="detect")
    metrics = model.val(
        data=str(DATA_YAML),
        split="val",
        batch=1,
        device=device,
        conf=0.001,
        workers=0,
        plots=False,
        save_json=False,
        project=str(OUT / "validation"),
        name=name,
        exist_ok=True,
        **COMMON,
        **HEAD_ARGS,
        **precision_args(fp16)
    )
    
    return {
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map)
    }

In [ ]:
def run_case(name, path, device="cpu", fp16=False):
    model, kwargs = prepare_model(path, device, fp16)
    latency = measure_latency(model, kwargs, name)
    
    del model
    gc.collect()

    if device != "cpu":
        torch.cuda.empty_cache()

    accuracy = measure_accuracy(path, name, device, fp16)
    return {
        "backend": name,
        **latency,
        **accuracy
    }

In [16]:
cpu_rows = []

cpu_rows.append(run_case("cpu_pytorch_fp32", MODEL_PT))
cpu_rows.append(run_case("cpu_onnx_fp32", MODEL_ONNX))

cpu_table = pd.DataFrame(cpu_rows)
cpu_table.to_csv(OUT / "cpu_comparison.csv", index=False)

display(cpu_table.round(4))

Device: cpu | FP16: False
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 84.2±21.1 MB/s, size: 72.0 KB)
val: Scanning /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/val/labels.cache... 387 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 387/387 85.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 387/387 5.9it/s 1:060.2sss
                   all        387        905      0.831      0.805      0.873      0.539
Speed: 1.6ms preprocess, 156.6ms inference, 0.0ms loss, 0.1ms postprocess per image
Loading /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Device: cpu | 

,backend,timed_calls,predict_mean_ms,predict_p95_ms,inference_mean_ms,mAP50,mAP50_95
0,cpu_pytorch_fp32,150,154.8732,224.2166,148.3940,0.8732,0.5388
1,cpu_onnx_fp32,150,131.5679,174.2188,124.1667,0.8732,0.5388


In [17]:
pt = cpu_table.set_index("backend").loc["cpu_pytorch_fp32"]
onnx = cpu_table.set_index("backend").loc["cpu_onnx_fp32"]

speedup = pt["predict_mean_ms"] / onnx["predict_mean_ms"]
map_change_pp = 100 * (onnx["mAP50_95"] - pt["mAP50_95"])

print(f"ONNX speedup: {speedup:.3f}x")
print(f"mAP50–95 change: {map_change_pp:+.3f} percentage points")

ONNX speedup: 1.177x
mAP50–95 change: +0.000 percentage points


In [12]:
gpu_rows = []

gpu_rows.append(
    run_case("gpu_pytorch_fp32", MODEL_PT, device=0, fp16=False)
)

gpu_rows.append(
    run_case("gpu_pytorch_fp16", MODEL_PT, device=0, fp16=True)
)

gpu_table = pd.DataFrame(gpu_rows)
gpu_table.to_csv(OUT / "gpu_comparison.csv", index=False)

display(gpu_table.round(4))

Device: cuda:0 | FP16: False
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 0.2±0.1 MB/s, size: 75.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/val/labels.cache... 387 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 387/387 58.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 387/387 1.0it/s 6:101.0ss
                   all        387        905      0.831      0.805      0.873      0.539
Speed: 0.6ms preprocess, 11.5ms inference, 0.0ms loss, 0.3ms postprocess per image
Device: cuda:0 | FP16: True
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11

,backend,timed_calls,predict_mean_ms,predict_p95_ms,inference_mean_ms,mAP50,mAP50_95
0,gpu_pytorch_fp32,150,10.7419,12.7526,8.5016,0.8732,0.5388
1,gpu_pytorch_fp16,150,17.6093,22.3608,14.3730,0.8733,0.5375


In [13]:
gpu_rows = []

gpu_rows.append(
    run_case("gpu_pytorch_fp32", MODEL_PT, device=0, fp16=False)
)

gpu_rows.append(
    run_case("gpu_pytorch_fp16", MODEL_PT, device=0, fp16=True)
)

gpu_table = pd.DataFrame(gpu_rows)
gpu_table.to_csv(OUT / "gpu_comparison.csv", index=False)

display(gpu_table.round(4))

Device: cuda:0 | FP16: False
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 113.4±32.2 MB/s, size: 77.8 KB)
val: Scanning /content/drive/MyDrive/Vision/Yolo_road2022_dataset/RDD2022_China_MotorBike/val/labels.cache... 387 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 387/387 95.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 387/387 51.6it/s 7.5s0.0s
                   all        387        905      0.831      0.805      0.873      0.539
Speed: 0.5ms preprocess, 10.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Device: cuda:0 | FP16: True
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 

,backend,timed_calls,predict_mean_ms,predict_p95_ms,inference_mean_ms,mAP50,mAP50_95
0,gpu_pytorch_fp32,150,14.9433,19.3104,11.8174,0.8732,0.5388
1,gpu_pytorch_fp16,150,12.5977,15.1871,10.3499,0.8733,0.5375
